<h1>Seeds</h1>

In [ ]:
#%pip install -q --upgrade pip

In [ ]:
!pip install unsloth "xformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.7/293.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 37.6 MB/s et

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from huggingface_hub import login
login(token='##############')  # Replace with your actual token

In [ ]:
#from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "meta-llama/Meta-Llama-3-8B-instruct"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
import pandas as pd
import numpy as np
import random

In [ ]:
def generate_seeds(num_seeds=20, seed=42):
    """Generates a list of random seeds.

    Args:
        num_seeds: The number of seeds to generate.
        seed: The initial seed for the random number generator (for reproducibility).

    Returns:
        A list of random integer seeds.
    """
    random.seed(seed)  # Set initial seed for reproducibility
    seeds = [random.randint(1, 100000) for _ in range(num_seeds)]
    return seeds

In [ ]:
seeds = generate_seeds(num_seeds=32)

#timeline

In [ ]:
timeline = pd.read_csv('/content/drive/MyDrive/centaur/horizon/timeline_structure.csv')

In [ ]:
timeline

,participant_id,block,game,gameLength,info_condition,reward_mean_H,reward_mean_I,trial_num_block,choice,reward,type,reward_H,reward_I,horizon
0,1,1,1,5,3.0,40.0,70.0,1,I,66.0,forced,44.0,66.0,1
1,1,1,1,5,3.0,40.0,70.0,2,I,80.0,forced,47.0,80.0,1
2,1,1,1,5,3.0,40.0,70.0,3,H,29.0,forced,29.0,62.0,1
3,1,1,1,5,3.0,40.0,70.0,4,I,75.0,forced,45.0,75.0,1
4,1,1,1,5,3.0,40.0,70.0,5,NaN,NaN,free,33.0,81.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74395,31,4,320,5,2.0,40.0,10.0,1,I,1.0,forced,44.0,1.0,1
74396,31,4,320,5,2.0,40.0,10.0,2,I,1.0,forced,47.0,1.0,1
74397,31,4,320,5,2.0,40.0,10.0,3,H,51.0,forced,51.0,0.0,1
74398,31,4,320,5,2.0,40.0,10.0,4,H,34.0,forced,34.0,8.0,1


# Llama Initialisation

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No NVIDIA GPU detected.")

CUDA available: True
GPU name: Tesla T4


In [ ]:
import transformers
def create_text_generation_pipeline(model, tokenizer, temperature=1.0):
    """
    Creates a text-generation pipeline with the given model and tokenizer.

    Args:
        model: The preloaded model for text generation.
        tokenizer: The corresponding tokenizer.
        temperature (float): Sampling temperature for generation (default: 1.0).
        max_new_tokens (int): Maximum number of tokens to generate (default: 1024).

    Returns:
        A transformers pipeline object for text generation.
    """
    return transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            trust_remote_code=True,
            pad_token_id=0,
            do_sample=True,
            temperature=1.0,
            max_new_tokens=1,
    )

# Example usage:
# pipe = create_text_generation_pipeline(model, tokenizer)


In [ ]:
import json
import re

def extract_model_choice(raw_response: str) -> str:
    """
    Extracts choice ('I' or 'H') from model's raw response text.
    Handles JSON and loose formats robustly.
    """
    try:
        # First try direct JSON parsing
        response_data = json.loads(raw_response)
        choice = response_data.get("choice", "").strip().upper()
        if choice in {"I", "H"}:
            return choice

    except json.JSONDecodeError:
        # Fallback: Search for JSON pattern in text
        json_match = re.search(r'{\s*"choice"\s*:\s*"?(I|H)"?\s*}', raw_response, re.IGNORECASE)
        if json_match:
            response_data = json.loads(json_match.group().replace("'", '"'))  # normalize quotes
            choice = response_data.get("choice", "").strip().upper()
            if choice in {"I", "H"}:
                return choice

    # Final fallback: Find first standalone I or H
    char_match = re.search(r'\b[IiHh]\b', raw_response)
    if char_match:
        return char_match.group().upper()

    raise ValueError("No valid choice ('I' or 'H') found in model response")


In [ ]:
def generate(prompt, pipe):
    # Convert the prompt list to a single string
    prompt_items = [str(item) if not isinstance(item, str) else item for item in prompt]
    prompt_str = "".join(prompt_items)
    pipe(prompt_str)
    return pipe(prompt_str)[0]['generated_text'][len(prompt_str):]

In [ ]:
def format_forced_trials(forced_df) -> str:
    """Format forced (instructed) trials as readable prompt text."""
    lines = []
    for idx, (_, row) in enumerate(forced_df.iterrows(), start=1):
        lines.append(f"Trial {idx} (instructed): You were instructed to press {row['choice']} and received {row['reward']} points.")
    return "\n".join(lines)


In [ ]:
def format_past_trials(past_df):
    """Format past free-choice trials for the prompt."""
    trials_text = []
    for _, row in past_df.iterrows():
        trials_text.append(f"Trial {row['trial_num_block']}(free): You chose <<{row['choice']}>> and get {row['reward']} points.")
    return trials_text

In [ ]:
import random
from collections import defaultdict

def build_slot_prompt_llama(
    current_trial: int,
    past_trials: list,
    forced_choices: str,
    total_trials: int,
    game_number,
    total_games=320
) -> str:
    """
    Builds a slot task prompt that:
    1. Preserves task structure
    2. Minimizes positional bias
    3. Encourages evidence-based exploration
    4. Helps model infer the optimal choice
    """

    # Randomize machine labels each game to avoid bias
    machines = ["H", "I"]
    random.shuffle(machines)
    label_a, label_b = machines

    # Format forced lines for outcome summary calculation


    # Show recent free choices
    recent_history_text = "\n".join(past_trials[-5:]) if past_trials else "No free choices yet."

    return f"""<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>

You are participating in multiple games ({total_games} total) involving two slot machines, labeled I and H.
The two slot machines change between games. Each game begins with four trials where you are instructed which machine to choose.

**Game Structure:**
1. First 4 trials: Instructed choices
2. After these, you make your own choices for several trials
3. Then the game ends, and a new one begins

**Important Rules:**
- Each machine gives variable points when you choose it
- Pay close attention to outcomes during the instructed trials
- These early results may help you estimate which machine is better
- Your goal is to maximize points across all games, even when uncertain
- Machine labels (I and H) are randomized every game

<|eot_id|>

<|start_header_id|>user<|end_header_id|>
# Game {game_number} of {total_games} | Trial {current_trial} of {total_trials}

## Instructed Trials
{forced_choices}



## Recent Free Choices
{recent_history_text}

## Think Before You Choose
- Which machine gave more points during the forced trials?
- Was one clearly better, or are you uncertain?
- Use this evidence to guide your next choice

## Your Decision
Which machine will you choose NEXT to maximize your points?

<|response_format|>
{{"choice": "{label_a}"}}  // OR {{"choice": "{label_b}"}}

<|critical_instructions|>
- Respond with VALID JSON ONLY
- Choose only "I" or "H" as your answer
- Base your decision on observed rewards

<|eot_id|>
"""


In [ ]:
import torch
import random
import transformers

def fix_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    transformers.set_seed(seed)  # For Hugging Face models

In [ ]:
timeline

,participant_id,block,game,gameLength,info_condition,reward_mean_H,reward_mean_I,trial_num_block,choice,reward,type,reward_H,reward_I,horizon
0,1,1,1,5,3.0,40.0,70.0,1,I,66.0,forced,44.0,66.0,1
1,1,1,1,5,3.0,40.0,70.0,2,I,80.0,forced,47.0,80.0,1
2,1,1,1,5,3.0,40.0,70.0,3,H,29.0,forced,29.0,62.0,1
3,1,1,1,5,3.0,40.0,70.0,4,I,75.0,forced,45.0,75.0,1
4,1,1,1,5,3.0,40.0,70.0,5,NaN,NaN,free,33.0,81.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74395,31,4,320,5,2.0,40.0,10.0,1,I,1.0,forced,44.0,1.0,1
74396,31,4,320,5,2.0,40.0,10.0,2,I,1.0,forced,47.0,1.0,1
74397,31,4,320,5,2.0,40.0,10.0,3,H,51.0,forced,51.0,0.0,1
74398,31,4,320,5,2.0,40.0,10.0,4,H,34.0,forced,34.0,8.0,1


In [ ]:
def simulate_participant_by_block(timeline_df, pipe, participant_id, seeds):
    """Simulates a participant's choices and rewards in the slot machine task."""
    participant_seed = seeds[int(participant_id - 1)]
    fix_seed(participant_seed)
    all_rows = []  # Final results will be collected here

    # Filter timeline for this participant
    participant_df = timeline_df[timeline_df['participant_id'] == participant_id]

    for block_num in sorted(participant_df['block'].unique()):
        block_df = participant_df[participant_df['block'] == block_num]

        for game in sorted(block_df['game'].unique()):
            game_df = block_df[block_df['game'] == game].sort_values('trial_num_block')

            # === extract metadata
            reward_means = {
                "H": game_df["reward_mean_H"].iloc[0],
                "I": game_df["reward_mean_I"].iloc[0]
            }
            horizon = game_df["horizon"].iloc[0]
            info_condition = game_df["info_condition"].iloc[0]

            forced_df = game_df[game_df["type"] == "forced"]
            free_df = game_df[game_df["type"] == "free"]

            cumulative_reward = forced_df["reward"].sum()

            # Start building simulated game DataFrame with forced trials
            simulated_game_df = forced_df.copy()

            # Append forced trials to all_rows
            for _, row in forced_df.iterrows():
                all_rows.append({
                    "participant_id": participant_id,
                    "block": block_num,
                    "game": game,
                    "horizon": horizon,
                    "info_condition": info_condition,
                    "reward_mean_H": reward_means["H"],
                    "reward_mean_I": reward_means["I"],
                    "trial_num": row["trial_num_block"],
                    "choice": row["choice"],
                    "reward": row["reward"],
                    "cumulative_reward": cumulative_reward,
                    "is_free": False
                })

            # Simulate free-choice trials
            for _, row in free_df.iterrows():
                current_trial = row["trial_num_block"]
                game=row['game']
                m1=free_df[free_df['trial_num_block']==current_trial]['reward_mean_H'].values[0]
                m2=free_df[free_df['trial_num_block']==current_trial]['reward_mean_I'].values[0]
                past_forced_df = simulated_game_df[simulated_game_df["type"] == "forced"]
                past_free_df = simulated_game_df[
                    (simulated_game_df["type"] == "free") &
                    (simulated_game_df["trial_num_block"] < current_trial)
                ]
                forced_trials_text = format_forced_trials(past_forced_df)
                free_trials_text = format_past_trials(past_free_df)

                prompt = build_slot_prompt_llama(
                      current_trial=current_trial,
                      past_trials=free_trials_text,
                      forced_choices=forced_trials_text,
                      total_trials=len(game_df),
                      game_number=game
                  )

                model_choice = generate(prompt, pipe)
                if m1>m2 and model_choice=="H":
                  is_optimal=True
                elif m1<m2 and model_choice=="I":
                  is_optimal=True
                else:
                  is_optimal=False

                # Update cumulative reward
                reward_h = row["reward_H"]
                reward_i = row["reward_I"]

                reward = reward_h if model_choice == "H" else reward_i if model_choice == "I" else None
                cumulative_reward += reward

                #print(f"this is model choice {model_choice} and it is reward{cumulative_reward}")

                # Create new simulated row and append to local game history
                simulated_row = row.copy()
                simulated_row["choice"] = model_choice
                simulated_row["reward"] = reward
                simulated_row["cumulative_reward"] = cumulative_reward
                simulated_row["prompt"] = "".join(str(p) for p in prompt)

                simulated_game_df = pd.concat([simulated_game_df, pd.DataFrame([simulated_row])])

                all_rows.append({
                    "participant_id": participant_id,
                    "block": block_num,
                    "game": game,
                    "horizon": horizon,
                    "info_condition": info_condition,
                    "reward_mean_H": reward_means["H"],
                    "reward_mean_I": reward_means["I"],
                    "trial_num": simulated_row["trial_num_block"],
                    "choice": model_choice,
                    "reward": reward,
                    "cumulative_reward": cumulative_reward,
                    "prompt": simulated_row["prompt"],
                    "is_free": True
                })

                #print(f"Prompt for trial {current_trial}:")
                #print("".join(prompt))
                print(f"Model choice: {model_choice} and it is optimal {is_optimal}")
                #print(f"Reward: {reward}")
                #print("---")

    print(f"✅ Simulated participant {participant_id}.")
    return pd.DataFrame(all_rows)


<h3>testing(can be skipped)</h3>

In [ ]:
# Create a copy of the timeline with 'game' renamed to 'game_number' if needed
#timeline_test = timeline[:100].copy()
# Run the simulation for participant 1
#result_test_4 = simulate_participant_by_block(timeline_test, pipe, 1, seeds)

In [ ]:
#result_test_4
#model_choice=result_test_4[result_test_4['is_free']==True]
#model_choice['choice'].value_counts()


In [ ]:
# Check if the sentence "you press and you get" exists in any prompt in result_test_4
#contains_sentence = result_test_4['prompt'].dropna().str.contains("you press <<H>>", case=False).any()
#print("Sentence found:", contains_sentence)

In [ ]:
# Print the prompt used for model choice in the first free trial of result_test_4
#first_free_trial = result_test_4[result_test_4['is_free'] == True][:100]
#for _, row in first_free_trial.iterrows():
    #print(f"Trial {row['trial_num_block']}:")
    #print("Prompt:", row['prompt'])
    #print("Model choice:", row['choice'])
    #print("Reward:", row['reward'])
    #print("Cumulative reward:", row['cumulative_reward'])

<h3>Centaur run </h3>

In [ ]:
import gc
import torch

In [ ]:
participant_ids = timeline['participant_id'].unique()
participant_ids

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [ ]:
all_results = []

for participant_id in participant_ids:
    print(f"\n🧠 Simulating participant {participant_id}")

    # 🔁 Re-initialize model and pipeline for each participant
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Re-initialize the model and tokenizer
    # Set the seed for reproducibility
    # Use the seed corresponding to the participant_id


    seed_id= seeds[int(participant_id - 1)]
    fix_seed(seed_id)

    ### LLM ###
    model,tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    model._past = None
    pipe = create_text_generation_pipeline(model, tokenizer)

    # Run participant simulation
    participant_data = timeline[timeline['participant_id'] == participant_id]
    result = simulate_participant_by_block(participant_data, pipe, participant_id,seeds)
    all_results.append(result)

    # Optional: cleanup
    del model, tokenizer, pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



🧠 Simulating participant 1
==((====))==  Unsloth 2025.6.9: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device set to use cuda:0


TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'

In [ ]:
all__results = pd.concat(all_results, ignore_index=True)

In [ ]:
all__results

In [ ]:
all_results_df =pd.DataFrame(all__results)

In [ ]:
all_results_df.to_csv("/content/drive/MyDrive/centaur/horizon_llama_one_participant.csv", index=False)